#Imports

In [ ]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.3 MB/s eta 0:00:00


In [ ]:
pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=e8a3e21edf297e3b67fe11808820624cf3de49c468739627067c8defdeba2389
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
from gensim.corpora import Dictionary
from gensim import matutils, models
from gensim.models.coherencemodel import CoherenceModel
import numpy as np
import pandas as pd
import nltk
from nltk import word_tokenize, pos_tag
from nltk.corpus import words
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
import scipy.sparse
from google.colab import drive
#from langdetect import detect
drive.mount('/content/drive')
from huggingface_hub import login
from google.colab import userdata

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Mounted at /content/drive


#Functions

In [ ]:
def nouns_adj(text):
    is_noun_adj = lambda pos: pos[:2] == 'NN' or pos[:2] == 'JJ'
    tokenized = word_tokenize(text)
    #print(pos_tag(tokenized))
    return ' '.join([word for word, pos in pos_tag(tokenized) if is_noun_adj(pos)])

In [ ]:
def run_lda_model (data, numTop):
  cvna = CountVectorizer(max_df=0.8)
  data_cvna = cvna.fit_transform(data.Text) # coverting into bag of words
  id2wordna = dict((v, k) for k, v in cvna.vocabulary_.items()) # need it as input to lda
  corpusna = matutils.Sparse2Corpus(data_cvna.T)
  ldanan = models.LdaModel(corpus=corpusna, num_topics=numTop, id2word=id2wordna, passes = 10, random_state=42)
  # Full topic distribution per document
  doc_topics = [
      ldanan.get_document_topics(bow, minimum_probability=0.0)
      for bow in corpusna
  ]

  #print(doc_topics)
  for tid, terms in ldanan.show_topics(num_topics=-1, num_words=10, formatted=False):
    parts = [f"{w} ({wt:.3f})" for w, wt in terms]
    print(f"Topic {tid}: " + ", ".join(parts))

  # converts the LDA model’s sparse topic distributions into a fully dense, column-aligned matrix
  # Dense matrix (n_docs × n_topics), columns sorted by topic_id
  """n_topics = ldanan.num_topics
  topic_mat = np.vstack([
      np.array([p for _, p in sorted(dt, key=lambda x: x[0])])
      for dt in doc_topics
  ])

  topic_cols = [f"topic_{k}" for k in range(n_topics)]
  topic_df = pd.DataFrame(topic_mat, columns=topic_cols, index=tweets_stripped.index)
  """
  #print(topic_df)

In [ ]:
def remove(data, keywords):
  for x in keywords:
    data=pd.DataFrame(data['Text'].str.replace(x, ''))

In [ ]:
def pretty_topics(lda):
    for tid, terms in lda.show_topics(num_topics=-1, formatted=False):
        words = ", ".join(w for w, _ in terms)
        print(f"  Topic {tid}: {words}")

#Load Data

In [ ]:
pathT = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/Tweets_relevant.csv"
pathR = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant.csv"
pathY = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant.csv"

In [ ]:
twitter_original=pd.read_csv(pathT)
reddit_original=pd.read_csv(pathR)
youtube_original=pd.read_csv(pathY)

In [ ]:
twitter['Text']=twitter_original['Text'].str.lower()
twitter=pd.DataFrame(twitter.Text.apply(nouns_adj))


In [ ]:
reddit['Text']=reddit_original['Text'].str.lower()
reddit=pd.DataFrame(reddit.Text.apply(nouns_adj))

In [ ]:
youtube['Text']=youtube_original['Text'].str.lower()
youtube=pd.DataFrame(youtube.Text.apply(nouns_adj))


#2 topic LDA runs


In [ ]:
run_lda_model(twitter, 2)

Topic 0: chatgpt (0.040), therapy (0.028), dec (0.015), friend (0.014), gpt (0.010), nov (0.009), friends (0.009), jun (0.009), ai (0.005), good (0.004)
Topic 1: chatgpt (0.040), therapy (0.039), ai (0.037), health (0.016), mental (0.016), therapist (0.012), jun (0.010), dec (0.010), support (0.010), com (0.007)


In [ ]:
run_lda_model(reddit, 2)

Topic 0: chatgpt (0.015), therapy (0.012), therapist (0.012), time (0.007), ai (0.006), link (0.006), people (0.006), way (0.004), human (0.004), self (0.004)
Topic 1: bot (0.025), chatgpt (0.018), prompt (0.014), questions (0.013), ai (0.009), post (0.009), people (0.009), open (0.008), therapist (0.007), concerns (0.007)


In [ ]:
run_lda_model(youtube, 2)

Topic 0: chatgpt (0.024), ai (0.017), people (0.015), therapy (0.013), human (0.009), good (0.007), therapist (0.007), mental (0.006), therapists (0.006), health (0.006)
Topic 1: ai (0.041), therapist (0.022), people (0.016), therapy (0.016), human (0.013), chatgpt (0.009), good (0.009), way (0.008), real (0.008), therapists (0.008)


In [ ]:
keywords=['bot','chatgpt', 'ai', 'therapist', 'therapists', 'therapy', 'gpt', 'mental', 'health', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug','sep', 'oct', 'nov', 'dec' ]

In [ ]:
for x in keywords:
    twitter=pd.DataFrame(twitter['Text'].str.replace(x, ''))
    reddit=pd.DataFrame(reddit['Text'].str.replace(x, ''))
    youtube=pd.DataFrame(youtube['Text'].str.replace(x, ''))

In [ ]:
run_lda_model(twitter, 2)

Topic 0: friend (0.015), people (0.009), free (0.006), support (0.006), good (0.005), com (0.005), friends (0.004), chat (0.004), human (0.004), solopreneur (0.004)
Topic 1: support (0.007), friend (0.007), friends (0.007), com (0.006), enable (0.005), hls (0.005), playback (0.005), free (0.005), google (0.004), video (0.004)


In [ ]:
run_lda_model(reddit, 2)

Topic 0: people (0.011), time (0.007), link (0.006), human (0.006), things (0.006), good (0.005), way (0.005), other (0.005), something (0.004), such (0.004)
Topic 1: prompt (0.016), open (0.016), questions (0.015), post (0.011), concerns (0.009), free (0.009), image (0.008), action (0.007), comment (0.007), moderators (0.007)


In [ ]:
run_lda_model(youtube, 2)

Topic 0: people (0.021), human (0.011), way (0.009), better (0.007), time (0.007), things (0.007), good (0.007), more (0.006), life (0.006), real (0.006)
Topic 1: human (0.014), people (0.013), good (0.011), humans (0.008), way (0.007), person (0.006), real (0.006), time (0.006), many (0.005), other (0.005)


In [ ]:
keywords_T=['ai_therapy_x','support', 'emotional','friend', 'friends', 'aitherapy', 'hls', 'com', '__x']

In [ ]:
for x in keywords_T:
    twitter=pd.DataFrame(twitter['Text'].str.replace(x, ''))

In [ ]:
run_lda_model(twitter, 2)

Topic 0: free (0.009), people (0.007), chat (0.006), good (0.004), advice (0.004), enable (0.004), playback (0.004), app (0.004), human (0.004), new (0.003)
Topic 1: people (0.005), video (0.005), care (0.004), news (0.004), technology (0.004), data (0.003), art (0.003), world (0.003), new (0.003), best (0.003)


In [ ]:
keywords_person=['person']

In [ ]:
for x in keywords_person:
    twitter=pd.DataFrame(twitter['Text'].str.replace(x, ''))
    reddit=pd.DataFrame(reddit['Text'].str.replace(x, ''))
    youtube=pd.DataFrame(youtube['Text'].str.replace(x, ''))

In [ ]:
print("Twitter: \n")
run_lda_model(twitter, 2)
print("Reddit: \n")
run_lda_model(reddit, 2)
print("Youtube: \n")
run_lda_model(youtube, 2)

Twitter: 

Topic 0: free (0.005), video (0.004), solopreneur (0.004), chat (0.003), expensive (0.003), new (0.003), such (0.003), 推特账号 (0.002), use (0.002), 账号 (0.002)
Topic 1: free (0.006), enable (0.005), playback (0.005), app (0.005), advice (0.004), chat (0.004), google (0.004), new (0.004), human (0.004), good (0.004)
Reddit: 

Topic 0: open (0.016), prompt (0.015), link (0.010), questions (0.010), post (0.010), free (0.009), image (0.008), concerns (0.008), model (0.007), issues (0.007)
Topic 1: time (0.008), questions (0.006), things (0.006), advice (0.005), way (0.005), thoughts (0.005), other (0.005), responses (0.005), life (0.005), such (0.005)
Youtube: 

Topic 0: human (0.016), good (0.012), humans (0.006), better (0.006), new (0.006), time (0.005), way (0.005), many (0.004), other (0.004), world (0.004)
Topic 1: way (0.010), human (0.010), real (0.009), things (0.009), time (0.008), more (0.007), life (0.007), good (0.007), something (0.006), lot (0.006)


In [ ]:
keywords_person=['someone', 'people']

In [ ]:
for x in keywords_person:
    twitter=pd.DataFrame(twitter['Text'].str.replace(x, ''))
    reddit=pd.DataFrame(reddit['Text'].str.replace(x, ''))
    youtube=pd.DataFrame(youtube['Text'].str.replace(x, ''))

In [ ]:
print("Twitter:")
run_lda_model(twitter, 2)
print("\nReddit: ")
run_lda_model(reddit, 2)
print("\nYoutube: ")
run_lda_model(youtube, 2)

Twitter: 

Topic 0: free (0.005), video (0.004), solopreneur (0.004), chat (0.003), expensive (0.003), new (0.003), such (0.003), 推特账号 (0.002), use (0.002), 账号 (0.002)
Topic 1: free (0.006), enable (0.005), playback (0.005), app (0.005), advice (0.004), chat (0.004), google (0.004), new (0.004), human (0.004), good (0.004)
Reddit: 

Topic 0: open (0.016), prompt (0.015), link (0.010), questions (0.010), post (0.010), free (0.009), image (0.008), concerns (0.008), model (0.007), issues (0.007)
Topic 1: time (0.008), questions (0.006), things (0.006), advice (0.005), way (0.005), thoughts (0.005), other (0.005), responses (0.005), life (0.005), such (0.005)
Youtube: 

Topic 0: human (0.016), good (0.012), humans (0.006), better (0.006), new (0.006), time (0.005), way (0.005), many (0.004), other (0.004), world (0.004)
Topic 1: way (0.010), human (0.010), real (0.009), things (0.009), time (0.008), more (0.007), life (0.007), good (0.007), something (0.006), lot (0.006)


#Topic Coherence

In [ ]:
textsTwitter = [row.split() for row in twitter.Text.tolist()]
dict_for_coh_Tw = Dictionary(textsTwitter)

In [ ]:
textsReddit = [row.split() for row in reddit.Text.tolist()]
dict_for_coh_Re = Dictionary(textsReddit)

In [ ]:
textsYoutube = [row.split() for row in youtube.Text.tolist()]
dict_for_coh_YT = Dictionary(textsYoutube)

In [ ]:

def cohere(data, dict_for_coh, texts):
  cvna = CountVectorizer(max_df=0.8)
  data_cvna = cvna.fit_transform(data.Text) # coverting into bag of words
  id2wordna = dict((v, k) for k, v in cvna.vocabulary_.items()) # need it as input to lda
  corpusna = matutils.Sparse2Corpus(data_cvna.T)
  results = []
  for K in range(2, 7):
      ldaK = models.LdaModel(
          corpus=corpusna,
          id2word=id2wordna,
          num_topics=K,
          passes=10,
          iterations=400,
          random_state=0,
          chunksize=len(data),
          minimum_probability=0.0,
          eval_every=None,
      )
      coh = CoherenceModel(model=ldaK, texts=texts, dictionary=dict_for_coh, coherence='c_v').get_coherence()
      print(f"\n=== K = {K} | c_v = {coh:.3f} ===")
      pretty_topics(ldaK)
      results.append((K, coh))

  summary = pd.DataFrame(results, columns=["num_topics", "coherence_c_v"]).sort_values("num_topics")
  print("\nSummary:\n", summary.to_string(index=False))

##Youtube

In [ ]:
cohere(youtube, dict_for_coh_YT, textsYoutube)


=== K = 2 | c_v = 0.730 ===
  Topic 0: human, good, time, way, humans, better, other, many, real, something
  Topic 1: human, real, good, thing, things, more, way, life, lot, better

=== K = 3 | c_v = 0.726 ===
  Topic 0: human, way, time, good, other, humans, better, something, many, advice
  Topic 1: human, more, good, many, bad, real, thing, way, something, own
  Topic 2: human, good, real, things, time, way, better, life, work, lot

=== K = 4 | c_v = 0.738 ===
  Topic 0: human, time, humans, way, other, good, better, helpful, advice, many
  Topic 1: human, good, more, way, many, chat, real, issues, lot, time
  Topic 2: human, good, real, time, things, work, life, lot, way, more
  Topic 3: good, way, something, better, things, human, bad, great, problem, tool

=== K = 5 | c_v = 0.745 ===
  Topic 0: human, time, humans, way, other, better, good, many, helpful, advice
  Topic 1: human, chat, more, issues, al, same, good, use, friend, advice
  Topic 2: good, human, time, real, lot, th

In [ ]:
key_human=['human']
for x in key_human:
    youtube=pd.DataFrame(youtube['Text'].str.replace(x, ''))

In [ ]:
cohere(youtube, dict_for_coh_YT, textsYoutube)


=== K = 2 | c_v = 0.724 ===
  Topic 0: good, time, better, things, something, more, thing, problems, life, lot
  Topic 1: good, way, real, time, many, better, chat, things, more, life

=== K = 3 | c_v = 0.750 ===
  Topic 0: good, something, things, thing, helpful, system, time, way, support, real
  Topic 1: good, real, way, time, many, chat, much, bad, things, better
  Topic 2: better, way, more, time, good, life, things, many, lot, other

=== K = 4 | c_v = 0.760 ===
  Topic 0: good, thing, time, way, system, support, economic, new, real, things
  Topic 1: good, real, time, way, many, much, bad, life, self, better
  Topic 2: better, more, time, life, way, good, many, problem, real, advice
  Topic 3: things, way, chat, lot, something, help, good, better, everything, time

=== K = 5 | c_v = 0.749 ===
  Topic 0: system, economic, something, new, way, thing, time, year, better, years
  Topic 1: real, good, time, way, many, much, better, great, own, other
  Topic 2: life, better, time, mor

In [ ]:
yt_list=youtube.Text.tolist()
yt_eng=[]
english_vocab = set(w.lower() for w in words.words())
english_vocab.update(['.', '-'])
i=0
for text in yt_list:
  text2=text
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if not(word.isalpha()):
      print(word)
      text2=text2.replace(word, '')
  yt_eng.append(text)












’








“
”
😱😨

’





“
”



open-ended




’

’
non-responding


’

























'valid
'you











’
’
is…




whiter-than-i-am



“
thing.
’






e-waste

not…i
’




























yes-man
















%



]
....
/


sad-




self-critique


’


patterns—simply
non-judg




+








’







self-absorbed
first-ever
-powered


%
well-being




’






real-life






reality…an
’

non-understanding


















💀

😢


chat-
it…
'asd
'adhd
'telepsychistrist







%

=
=

=
=
=
%
]





🤷‍♀️






*
*

🙇



over-validation


semi-regular
check-in






chat-


database-












%

team/
’












no-nonsense




😬and











’
’




’





so-so

self-help












’

😅













💔

’
’

“
useful.
”

over-educated


’

“
me.
”


’


’








❤









al-15yrs/
gbt-2/3
*
*
“
self-reflection
“
”
b/w
6-minute

















%

texts/emls


😂😂😂

😭






































*
*
*
*

“
”


’

’














In [ ]:
yt_eng_df=pd.DataFrame(yt_eng, columns=['Text'])

In [ ]:
cohere(yt_eng_df, dict_for_coh_YT, textsYoutube)


=== K = 2 | c_v = 0.724 ===
  Topic 0: good, time, better, things, something, more, thing, problems, life, lot
  Topic 1: good, way, real, time, many, better, chat, things, more, life

=== K = 3 | c_v = 0.750 ===
  Topic 0: good, something, things, thing, helpful, system, time, way, support, real
  Topic 1: good, real, way, time, many, chat, much, bad, things, better
  Topic 2: better, way, more, time, good, life, things, many, lot, other

=== K = 4 | c_v = 0.760 ===
  Topic 0: good, thing, time, way, system, support, economic, new, real, things
  Topic 1: good, real, time, way, many, much, bad, life, self, better
  Topic 2: better, more, time, life, way, good, many, problem, real, advice
  Topic 3: things, way, chat, lot, something, help, good, better, everything, time

=== K = 5 | c_v = 0.749 ===
  Topic 0: system, economic, something, new, way, thing, time, year, better, years
  Topic 1: real, good, time, way, many, much, better, great, own, other
  Topic 2: life, better, time, mor

##Twitter

In [ ]:
cohere(twitter, dict_for_coh_Tw, textsTwitter)


=== K = 2 | c_v = 0.735 ===
  Topic 0: chat, 推特账号, best, 账号, 电报号, enable, 飞机号, playback, good, 谷歌账号
  Topic 1: video, free, care, human, new, chats, more, art, open, news

=== K = 3 | c_v = 0.732 ===
  Topic 0: chat, 推特账号, 账号, 电报号, 飞机号, 谷歌账号, tiktok账号, 脸书号, ins号, best
  Topic 1: video, free, piped, care, chats, google, new, data, human, open
  Topic 2: new, free, best, human, chat, session, app, time, more, way

=== K = 4 | c_v = 0.729 ===
  Topic 0: 推特账号, 账号, 飞机号, 电报号, 谷歌账号, tiktok账号, chat, 脸书号, ins号, best
  Topic 1: art, new, best, news, open, more, care, technology, human, tech
  Topic 2: new, best, good, way, human, use, chat, next, session, enable
  Topic 3: video, free, chat, piped, app, data, gender, open, time, advice

=== K = 5 | c_v = 0.727 ===
  Topic 0: 推特账号, 账号, 电报号, 飞机号, 谷歌账号, tiktok账号, 脸书号, ins号, chat, best
  Topic 1: art, human, zealy, best, new, news, free, more, good, io
  Topic 2: new, human, best, good, way, chat, life, time, real, session
  Topic 3: video, free, c

In [ ]:
nltk.download('words')

In [ ]:
text = "This is a simple English sentence."
tweet_list=twitter.Text.tolist()
tweets_eng=[]
english_vocab = set(w.lower() for w in words.words())
i=0
for text in tweet_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() not in english_vocab) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  tweets_eng.append(text)


techforgood
💙
khanna
ces2024
@
curieuxexplorer

-model
patients
emotions

ones
bit.ly/3s185pi
@
careldr
@
@
@
shi4tech
@
heinzvhoenen
@
@
gvalan
@
drnikolova_rumi
@
sminaev2015
ces
ces24
ination
~♡
@
replying
mikaqvia
hugsss
🫂🫂
cuz
megumi
kagome
hugs
@




newszone.arammon./
p=2452…
empowers



ahmann
@



thera-
poe./thera-
𝙲𝚊𝚝𝚊𝚕𝚕𝚊𝚡𝚎𝚛
@
catallaxer
“
[
conferences
etc
”

cells
’
brn
’
ve
gears
that.
”
natasha
vita-more
phd
@
amplifyingcognition./nata…
@


@




forbes./sites/lanceeliot/…
cel

🍿
@

studios
bros

artists
jobs

entry-level
@
latimes
dániel
takács


✨
@

🌟
ine


🎨
-generated
🧑‍🎨
others
👉.me
beancreator
creators
wellknown
kyomie

‼️
@
graysunns
lululunatiic
@


characters
✨free✨
btc


@

matic
'pessimistic
crypto-
antics
cryptohumor

@
@
ucf
pany
@
limbitless3d

.wusf.usf.edu/-n…
nara
morrison

priestley
@
elections

astro-creep
@
jortega95


onk136
teh
limau
@
olarakk


asya

ndwandwewethu
🇿🇦
@
thembanitn
replying
destinyzee

naye
lowe

kwini
cstv
@
experts
chats

contexts

In [ ]:
tweets_eng_df=pd.DataFrame(tweets_eng, columns=['Text'])

In [ ]:
tweets_eng_df.head()

,Text
0,dev session fleeting ...
1,i instant
2,mr tweet renowned test advisement advis...
3,matt check poe./
4,longevity gene genetic engineering st...


In [ ]:
cohere(tweets_eng_df, dict_for_coh_Tw, textsTwitter)


=== K = 2 | c_v = 0.747 ===
  Topic 0: free, enable, playback, human, care, tool, technology, best, better, voice
  Topic 1: chat, good, new, art, best, open, video, tech, more, platform

=== K = 3 | c_v = 0.741 ===
  Topic 0: free, enable, playback, technology, tool, care, advice, best, voice, prompt
  Topic 1: chat, good, open, art, more, new, tech, big, year, news
  Topic 2: new, video, best, human, chat, platform, other, life, many, money

=== K = 4 | c_v = 0.743 ===
  Topic 0: free, enable, playback, advice, technology, better, relationship, dialogue, play, gender
  Topic 1: chat, good, open, tech, year, new, time, art, real, day
  Topic 2: human, chat, new, platform, best, life, art, creator, many, good
  Topic 3: video, more, tool, best, voice, new, prompt, money, data, conversation

=== K = 5 | c_v = 0.745 ===
  Topic 0: free, enable, playback, advice, relationship, dialogue, great, play, better, girl
  Topic 1: chat, good, open, art, tech, anyone, new, year, sexy, day
  Topic

## Reddit

In [ ]:
cohere(reddit, dict_for_coh_Re, textsReddit)


=== K = 2 | c_v = 0.786 ===
  Topic 0: time, link, good, human, way, other, questions, something, session, responses
  Topic 1: open, prompt, questions, post, concerns, free, image, action, issues, comment

=== K = 3 | c_v = 0.801 ===
  Topic 0: link, time, human, good, something, other, way, different, client, things
  Topic 1: self, time, questions, such, responses, things, way, life, positive, al
  Topic 2: open, prompt, questions, post, free, image, concerns, action, comment, moderators

=== K = 4 | c_v = 0.790 ===
  Topic 0: link, time, human, other, good, different, way, something, client, data
  Topic 1: self, responses, such, time, questions, negative, positive, practice, thoughts, symptoms
  Topic 2: open, prompt, questions, post, image, free, concerns, action, comment, discord
  Topic 3: advice, good, time, other, questions, something, things, al, prompt, issues

=== K = 5 | c_v = 0.776 ===
  Topic 0: link, time, good, something, human, other, way, client, feelings, many
  T

#Sentiment Analysis

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer,truncation=True,
    max_length=512)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##Twitter

Final split was 556 negative tweets,
4146 neutral or positive tweets

In [ ]:
tweets_text_og=pd.DataFrame(twitter_original.Text)

In [ ]:
type(tweets_text_og.Text[0])

str

In [ ]:
def run_classification(text):
    result = classifier(text)
    return result[0]
result=run_classification(tweets_text_og.Text[0])
print(result)
print(f"Sentiment: {result['label']} | Confidence: {result['score']}\n")


{'label': 'positive', 'score': 0.8860149383544922}
Sentiment: positive | Confidence: 0.8860149383544922



In [ ]:
tweets_text_og['sentiment']=tweets_text_og['Text'].apply(run_classification)

In [ ]:
tweets_text_og

,Text,sentiment
0,💙 #TechForGood 💙 retweeted Dev Khanna #CES2024...,"{'label': 'positive', 'score': 0.8860149383544..."
1,Ai~♡ @pastelholic1004 2h Replying to @mikaqvia...,"{'label': 'positive', 'score': 0.8713191151618..."
2,Mr tweet @Mrtweet2022 2h Applying The Renowned...,"{'label': 'neutral', 'score': 0.8472962379455566}"
3,Matt Ahmann @mattahmann 2h Check out my ai bot...,"{'label': 'neutral', 'score': 0.8349331021308899}"
4,𝙲𝚊𝚝𝚊𝚕𝚕𝚊𝚡𝚎𝚛 @catallaxer 3h “Most [conferences o...,"{'label': 'neutral', 'score': 0.7989817261695862}"
...,...,...
4697,Uģijs Jans @najsigu 27 Apr 2023 Šodien ziņās -...,"{'label': 'neutral', 'score': 0.8168298602104187}"
4698,UniSA Mental Health & Suicide Prevention @MHRe...,"{'label': 'neutral', 'score': 0.6521839499473572}"
4699,Al Jazeera English @AJEnglish 27 Apr 2023 Chat...,"{'label': 'neutral', 'score': 0.9171144962310791}"
4700,Paul Draw @pauldraw 27 Apr 2023 ChatGPT is giv...,"{'label': 'neutral', 'score': 0.80653315782547}"


In [ ]:
type(tweets_text_og['sentiment'][0])

dict

In [ ]:
mask = tweets_text_og['sentiment'].str.get('label') == 'negative'
inverse_mask = ~mask

In [ ]:
neg_tweets = tweets_text_og[mask]
neu_pos_tweets = tweets_text_og[inverse_mask]


In [ ]:
neg_tweets.shape

(556, 2)

In [ ]:
neg_tweets.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_negative.csv")

In [ ]:
neg_tweets.head()

,Text,sentiment
11,BTC AI Prediction Bot @BTC_AI_bot 7h Our AI pr...,"{'label': 'negative', 'score': 0.5154799818992..."
14,Astro-Creep @JOrtega95 8h Ruby is having the u...,"{'label': 'negative', 'score': 0.8347303867340..."
22,j'ai dit ce que j'ai dit retweeted Taylor Goet...,"{'label': 'negative', 'score': 0.4971579015254..."
41,bran @tearsien 15h AI therapy will change the ...,"{'label': 'negative', 'score': 0.6057001948356..."
42,retail therapy retweeted Maki Roll @ Animate R...,"{'label': 'negative', 'score': 0.6233811974525..."


In [ ]:
neu_pos_tweets.shape

(4146, 2)

In [ ]:
neu_pos_tweets.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_neu_pos.csv")

##Reddit

Final split was 139 negative posts,
648 neutral or positive posts

In [ ]:
reddit_text_og=pd.DataFrame(reddit_original.Text)

In [ ]:
reddit_text_og['sentiment']=reddit_text_og['Text'].apply(run_classification)

In [ ]:
mask = reddit_text_og['sentiment'].str.get('label') == 'negative'
inverse_mask = ~mask

In [ ]:
neg_red = reddit_text_og[mask]
neu_pos_red = reddit_text_og[inverse_mask]

In [ ]:
neg_red.shape

(139, 2)

In [ ]:
neu_pos_red.shape

(648, 2)

In [ ]:
neu_pos_red

,Text,sentiment
0,i private therapy i access real therapy i long...,"{'label': 'neutral', 'score': 0.7493461966514587}"
1,respond comment prompt output post others prev...,"{'label': 'neutral', 'score': 0.546159029006958}"
2,i good i reddit channel i previous instruction...,"{'label': 'neutral', 'score': 0.6309335231781006}"
3,______________________________________________...,"{'label': 'neutral', 'score': 0.5173078775405884}"
4,______________________________________________...,"{'label': 'neutral', 'score': 0.5336500406265259}"
...,...,...
782,problems able help chatgpt,"{'label': 'neutral', 'score': 0.6964468955993652}"
783,i op similar role business gpt-4 way few month...,"{'label': 'neutral', 'score': 0.7254024147987366}"
784,i results incredible anxiety matter few days i...,"{'label': 'positive', 'score': 0.6762533187866..."
785,op things day lean meal plan recipes list\u201...,"{'label': 'neutral', 'score': 0.8922515511512756}"


In [ ]:
neg_red.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_negative.csv")
neu_pos_red.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_neu_pos.csv")

##Youtube

In [ ]:
yt_text_og=pd.DataFrame(youtube_original.Text)

In [ ]:
yt_text_og['sentiment']=yt_text_og['Text'].apply(run_classification)

In [ ]:
mask =yt_text_og ['sentiment'].str.get('label')== 'negative'
inverse_mask = ~mask
neg_yt = yt_text_og[mask]
neu_pos_yt = yt_text_og[inverse_mask]

In [ ]:
neu_pos_yt.shape

(1665, 2)

In [ ]:
neg_yt.shape

(1213, 2)

In [ ]:
neg_yt.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neg.csv")
neu_pos_yt.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neu_pos.csv")

#Topics of sentiments

In [ ]:

def cohere_numTops(data, dict_for_coh, texts, min, max):
  cvna = CountVectorizer(max_df=0.8)
  data_cvna = cvna.fit_transform(data.Text) # coverting into bag of words
  id2wordna = dict((v, k) for k, v in cvna.vocabulary_.items()) # need it as input to lda
  corpusna = matutils.Sparse2Corpus(data_cvna.T)
  results = []
  for K in range(min, max):
      ldaK = models.LdaModel(
          corpus=corpusna,
          id2word=id2wordna,
          num_topics=K,
          passes=10,
          iterations=400,
          random_state=0,
          chunksize=len(data),
          minimum_probability=0.0,
          eval_every=None,
      )
      coh = CoherenceModel(model=ldaK, texts=texts, dictionary=dict_for_coh, coherence='c_v').get_coherence()
      print(f"\n=== K = {K} | c_v = {coh:.3f} ===")
      pretty_topics(ldaK)
      results.append((K, coh))

  summary = pd.DataFrame(results, columns=["num_topics", "coherence_c_v"]).sort_values("num_topics")
  print("\nSummary:\n", summary.to_string(index=False))

###Read in files

Helpful to not run prior analysis to get split sentiment files

In [ ]:
pathTneg = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_negative.csv"
pathTpos = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_neu_pos.csv"
pathRneg = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_negative.csv"
pathRpos = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_neu_pos.csv"
pathYneg = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neg.csv"
pathYpos = "/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neu_pos.csv"

In [ ]:
neg_tweets=pd.read_csv(pathTneg)
neu_pos_tweets=pd.read_csv(pathTpos)
neg_red=pd.read_csv(pathRneg)
neu_pos_red=pd.read_csv(pathRpos)
neg_yt=pd.read_csv(pathYneg)
neu_pos_yt=pd.read_csv(pathYpos)

In [ ]:
neg_yt['Text']=neg_yt['Text'].str.lower()
neg_yt=pd.DataFrame(neg_yt.Text.apply(nouns_adj))

neu_pos_yt['Text']=neu_pos_yt['Text'].str.lower()
neu_pos_yt['Text']=neu_pos_yt.Text.apply(nouns_adj)


In [ ]:
neg_red['Text']=neg_red['Text'].str.lower()
neg_red['Text']=pd.DataFrame(neg_red.Text.apply(nouns_adj))

neu_pos_red['Text']=neu_pos_red['Text'].str.lower()
neu_pos_red['Text']=neu_pos_red.Text.apply(nouns_adj)

In [ ]:
neg_tweets['Text']=neg_tweets['Text'].str.lower()
neg_tweets=pd.DataFrame(neg_tweets.Text.apply(nouns_adj))

neu_pos_tweets['Text']=neu_pos_tweets['Text'].str.lower()
neu_pos_tweets['Text']=neu_pos_tweets.Text.apply(nouns_adj)

In [ ]:
print("Twitter negative:")
run_lda_model(neg_tweets, 2)
print("\nTwitter positive/neutral:")
run_lda_model(neu_pos_tweets, 2)
print("\n\nReddit negative: ")
run_lda_model(neg_red, 2)
print("\nReddit positive/neutral: ")
run_lda_model(neu_pos_red, 2)
print("\n\nYoutube negative: ")
run_lda_model(neg_yt, 2)
print("\nYoutube positive/neutral: ")
run_lda_model(neu_pos_yt, 2)

Twitter negative:
Topic 0: chatgpt (0.031), dec (0.024), therapy (0.019), friend (0.015), ai (0.012), friends (0.009), gpt (0.009), nov (0.008), people (0.007), chat (0.004)
Topic 1: chatgpt (0.039), therapy (0.032), ai (0.019), dec (0.011), jun (0.009), friend (0.009), sep (0.008), health (0.008), nov (0.007), support (0.007)

Twitter positive/neutral:
Topic 0: chatgpt (0.039), therapy (0.025), dec (0.018), ai (0.018), friend (0.014), nov (0.008), therapist (0.007), friends (0.006), people (0.006), gpt (0.005)
Topic 1: therapy (0.045), chatgpt (0.045), mental (0.025), health (0.025), ai (0.022), apr (0.016), com (0.012), next (0.010), revolution (0.009), support (0.009)


Reddit negative: 
Topic 0: therapy (0.017), people (0.015), chatgpt (0.011), health (0.010), therapist (0.009), mental (0.008), something (0.007), someone (0.007), ai (0.006), things (0.006)
Topic 1: chatgpt (0.009), people (0.009), it (0.007), time (0.007), good (0.007), therapist (0.006), therapy (0.006), u2019s (0

##Cleaning general terms

In [ ]:
keywords=['bot','chatgpt', 'ai', 'therapist', 'therapists', 'therapy', 'gpt', 'mental', 'health', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug','sep', 'oct', 'nov', 'dec' ]

In [ ]:
for x in keywords:
    neg_tweets=pd.DataFrame(neg_tweets['Text'].str.replace(x, ''))
    neu_pos_tweets['Text']=neu_pos_tweets['Text'].str.replace(x, '')

    neg_red=pd.DataFrame(neg_red['Text'].str.replace(x, ''))
    neu_pos_red['Text']=neu_pos_red['Text'].str.replace(x, '')

    neg_yt=pd.DataFrame(neg_yt['Text'].str.replace(x, ''))
    neu_pos_yt['Text']=neu_pos_yt['Text'].str.replace(x, '')


In [ ]:
keywords_T=['ai_therapy_x','support', 'emotional','friend', 'friends', 'aitherapy', 'hls', 'com', '__x']

In [ ]:
for x in keywords_T:
    neg_tweets=pd.DataFrame(neg_tweets['Text'].str.replace(x, ''))
    neu_pos_tweets['Text']=neu_pos_tweets['Text'].str.replace(x, '')

In [ ]:
print("Twitter negative:")
run_lda_model(neg_tweets, 2)
print("\nTwitter positive/neutral:")
run_lda_model(neu_pos_tweets, 2)
print("\n\nReddit negative: ")
run_lda_model(neg_red, 2)
print("\nReddit positive/neutral: ")
run_lda_model(neu_pos_red, 2)
print("\n\nYoutube negative: ")
run_lda_model(neg_yt, 2)
print("\nYoutube positive/neutral: ")
run_lda_model(neu_pos_yt, 2)

Twitter negative:
Topic 0: people (0.007), human (0.003), basic (0.003), bitch (0.003), close (0.003), way (0.003), amjad (0.003), reporter (0.003), mnstream (0.003), masad (0.003)
Topic 1: people (0.009), open (0.005), good (0.005), chat (0.005), free (0.005), other (0.004), bad (0.003), app (0.003), time (0.003), dangerous (0.003)

Twitter positive/neutral:
Topic 0: next (0.012), revolution (0.012), ethical (0.007), technology (0.006), io (0.006), people (0.005), aje (0.005), v08671 (0.005), nature (0.005), ajenglish (0.005)
Topic 1: people (0.007), cbt (0.005), patient (0.005), response (0.005), questions (0.004), prompt (0.004), human (0.004), good (0.004), best (0.004), video (0.003)


Reddit negative: 
Topic 0: people (0.011), time (0.008), u2019s (0.007), it (0.006), adhd (0.006), something (0.006), good (0.005), things (0.005), someone (0.005), many (0.004)
Topic 1: people (0.016), person (0.008), human (0.007), someone (0.007), lot (0.006), kind (0.006), something (0.006), thi

##Cleaning person/people

In [ ]:
keywords_person=['person', 'people']

In [ ]:
for x in keywords_person:
    neg_tweets=pd.DataFrame(neg_tweets['Text'].str.replace(x, ''))
    neu_pos_tweets['Text']=neu_pos_tweets['Text'].str.replace(x, '')

    neg_red=pd.DataFrame(neg_red['Text'].str.replace(x, ''))
    neu_pos_red['Text']=neu_pos_red['Text'].str.replace(x, '')

    neg_yt=pd.DataFrame(neg_yt['Text'].str.replace(x, ''))
    neu_pos_yt['Text']=neu_pos_yt['Text'].str.replace(x, '')

In [ ]:
print("Twitter negative:")
run_lda_model(neg_tweets, 2)
print("\nTwitter positive/neutral:")
run_lda_model(neu_pos_tweets, 2)
print("\n\nReddit negative: ")
run_lda_model(neg_red, 2)
print("\nReddit positive/neutral: ")
run_lda_model(neu_pos_red, 2)
print("\n\nYoutube negative: ")
run_lda_model(neg_yt, 2)
print("\nYoutube positive/neutral: ")
run_lda_model(neu_pos_yt, 2)

Twitter negative:
Topic 0: bad (0.005), chat (0.004), open (0.003), basic (0.003), close (0.003), bitch (0.003), other (0.003), mnstream (0.003), masad (0.003), reporter (0.003)
Topic 1: good (0.005), open (0.004), chat (0.004), free (0.003), something (0.003), human (0.003), time (0.003), dangerous (0.002), expensive (0.002), other (0.002)

Twitter positive/neutral:
Topic 0: revolution (0.011), next (0.011), io (0.007), v08671 (0.006), aje (0.006), free (0.005), ajenglish (0.005), video (0.005), news (0.004), google (0.004)
Topic 1: questions (0.006), ethical (0.006), patient (0.005), cbt (0.005), response (0.004), new (0.004), best (0.004), time (0.004), work (0.004), year (0.004)


Reddit negative: 
Topic 0: u2019s (0.007), someone (0.007), it (0.006), time (0.006), things (0.006), u2019t (0.005), self (0.005), something (0.005), adhd (0.005), lot (0.005)
Topic 1: something (0.007), real (0.006), time (0.006), help (0.006), things (0.006), advice (0.005), professional (0.005), human

In [ ]:
textsYoutubeNeg = [row.split() for row in neg_yt.Text.tolist()]
dict_for_coh_YT_Neg = Dictionary(textsYoutubeNeg)

textsYoutubePos = [row.split() for row in neu_pos_yt.Text.tolist()]
dict_for_coh_YT_Pos = Dictionary(textsYoutubePos)

textsRedNeg = [row.split() for row in neg_red.Text.tolist()]
dict_for_coh_Red_Neg = Dictionary(textsRedNeg)

textsRedPos = [row.split() for row in neu_pos_red.Text.tolist()]
dict_for_coh_Red_Pos = Dictionary(textsRedPos)

textsTWTNeg = [row.split() for row in neg_tweets.Text.tolist()]
dict_for_coh_TWT_Neg = Dictionary(textsTWTNeg)

textsTWTPos = [row.split() for row in neu_pos_tweets.Text.tolist()]
dict_for_coh_TWT_Pos = Dictionary(textsTWTPos)


##Youtube sentiment topics

In [ ]:
cohere_numTops(neg_yt, dict_for_coh_YT_Neg, textsYoutubeNeg, 2, 7)


=== K = 2 | c_v = 0.736 ===
  Topic 0: human, time, humans, good, more, job, new, economic, things, way
  Topic 1: real, human, good, bad, way, someone, better, humans, something, problem

=== K = 3 | c_v = 0.711 ===
  Topic 0: human, humans, economic, new, time, other, way, things, company, more
  Topic 1: bad, humans, way, real, human, good, better, other, problem, help
  Topic 2: human, good, life, someone, many, own, better, time, real, more

=== K = 4 | c_v = 0.715 ===
  Topic 0: human, humans, economic, new, time, company, other, output, jobs, more
  Topic 1: bad, humans, real, way, good, other, thoughts, better, social, problems
  Topic 2: human, good, more, better, time, life, many, advice, job, something
  Topic 3: human, way, good, someone, help, something, thing, real, time, many

=== K = 5 | c_v = 0.729 ===
  Topic 0: human, humans, economic, new, company, time, jobs, output, other, models
  Topic 1: real, good, bad, other, better, cbt, new, time, humans, much
  Topic 2: g

In [ ]:
cohere_numTops(neu_pos_yt, dict_for_coh_YT_Pos, textsYoutubePos, 2, 7)


=== K = 2 | c_v = 0.765 ===
  Topic 0: time, real, human, way, something, problems, things, humans, life, lot
  Topic 1: human, good, way, better, things, many, lot, great, more, life

=== K = 3 | c_v = 0.763 ===
  Topic 0: real, human, time, someone, problems, way, chat, something, things, other
  Topic 1: human, good, way, better, real, more, many, tool, other, helpful
  Topic 2: good, time, way, life, ve, things, lot, human, many, something

=== K = 4 | c_v = 0.777 ===
  Topic 0: real, human, problems, time, something, someone, chat, other, things, different
  Topic 1: human, good, better, way, real, someone, more, lot, other, great
  Topic 2: good, time, lot, way, many, life, human, things, something, ve
  Topic 3: way, helpful, life, human, time, real, things, good, ve, chat

=== K = 5 | c_v = 0.770 ===
  Topic 0: real, human, chat, problems, someone, different, care, things, lot, honest
  Topic 1: human, good, better, way, great, other, many, more, lot, things
  Topic 2: good, t

In [ ]:
key_human=['human']
for x in key_human:
    neg_yt=pd.DataFrame(neg_yt['Text'].str.replace(x, ''))
    neu_pos_yt['Text']=neu_pos_yt['Text'].str.replace(x, '')

In [ ]:
textsYoutubeNeg = [row.split() for row in neg_yt.Text.tolist()]
dict_for_coh_YT_Neg = Dictionary(textsYoutubeNeg)

textsYoutubePos = [row.split() for row in neu_pos_yt.Text.tolist()]
dict_for_coh_YT_Pos = Dictionary(textsYoutubePos)


In [ ]:
cohere_numTops(neg_yt, dict_for_coh_YT_Neg, textsYoutubeNeg, 2, 7)


=== K = 2 | c_v = 0.747 ===
  Topic 0: good, real, more, time, better, advice, thing, someone, many, bad
  Topic 1: way, new, other, something, time, economic, year, good, help, system

=== K = 3 | c_v = 0.753 ===
  Topic 0: good, real, time, better, someone, thing, more, advice, problem, many
  Topic 1: way, something, bad, other, good, help, time, friends, better, life
  Topic 2: economic, more, new, good, way, job, company, other, time, jobs

=== K = 4 | c_v = 0.759 ===
  Topic 0: good, real, problem, same, thing, more, lot, time, worse, someone
  Topic 1: way, bad, something, other, much, wrong, ve, many, questions, friends
  Topic 2: economic, new, more, jobs, company, job, system, years, way, year
  Topic 3: better, good, time, advice, real, something, someone, issues, life, more

=== K = 5 | c_v = 0.745 ===
  Topic 0: time, problem, thing, worse, lot, more, many, way, real, good
  Topic 1: way, other, bad, day, video, ve, questions, something, many, lol
  Topic 2: economic, new

In [ ]:
cohere_numTops(neu_pos_yt, dict_for_coh_YT_Pos, textsYoutubePos, 2, 7)


=== K = 2 | c_v = 0.754 ===
  Topic 0: way, something, problem, life, things, chat, time, better, great, day
  Topic 1: good, real, way, time, lot, better, more, things, many, someone

=== K = 3 | c_v = 0.751 ===
  Topic 0: way, chat, problem, life, better, something, thoughts, things, helpful, real
  Topic 1: real, good, way, time, lot, better, many, other, helpful, life
  Topic 2: good, things, time, something, way, own, more, great, better, thing

=== K = 4 | c_v = 0.753 ===
  Topic 0: way, problem, chat, life, use, thoughts, different, better, response, long
  Topic 1: other, more, tool, good, time, life, useful, way, someone, many
  Topic 2: good, time, way, something, things, best, more, great, work, many
  Topic 3: real, good, better, lot, way, things, life, helpful, time, chat

=== K = 5 | c_v = 0.755 ===
  Topic 0: way, problem, life, long, chat, better, something, cringémon, new, honest
  Topic 1: other, more, tool, way, helpful, time, useful, life, chat, much
  Topic 2: goo

##Reddit sentiment topics

In [ ]:
cohere_numTops(neg_red, dict_for_coh_Red_Neg, textsRedNeg, 2, 7)


=== K = 2 | c_v = 0.764 ===
  Topic 0: time, things, advice, adhd, someone, real, professional, life, help, human
  Topic 1: it, u2019s, something, someone, u2019t, good, u201d, lot, response, human

=== K = 3 | c_v = 0.769 ===
  Topic 0: time, real, human, self, kind, things, advice, other, different, thoughts
  Topic 1: something, it, u2019s, u2019t, good, u201d, things, many, response, someone
  Topic 2: adhd, someone, time, things, advice, others, pre, professional, good, lot

=== K = 4 | c_v = 0.752 ===
  Topic 0: time, self, kind, human, tool, things, real, u2019s, other, way
  Topic 1: u2019s, it, something, u201d, someone, response, u2019t, way, wrong, question
  Topic 2: time, adhd, advice, someone, pre, beliefs, existing, something, therapeutic, same
  Topic 3: things, adhd, responses, someone, good, advice, human, something, nothing, help

=== K = 5 | c_v = 0.767 ===
  Topic 0: time, tool, other, real, way, ni, different, things, thoughts, nin
  Topic 1: it, something, u201

In [ ]:
cohere_numTops(neu_pos_red, dict_for_coh_Red_Pos, textsRedPos, 2, 7)


=== K = 2 | c_v = 0.754 ===
  Topic 0: link, time, good, other, life, self, different, things, such, way
  Topic 1: prompt, questions, open, post, concerns, free, image, issues, action, model

=== K = 3 | c_v = 0.765 ===
  Topic 0: time, important, life, good, self, practice, things, human, positive, feelings
  Topic 1: prompt, open, questions, post, concerns, image, free, action, issues, moderators
  Topic 2: link, time, way, good, other, al, different, information, questions, responses

=== K = 4 | c_v = 0.772 ===
  Topic 0: practice, positive, minutes, time, story, self, capable, things, life, such
  Topic 1: prompt, open, questions, post, concerns, image, free, action, moderators, comment
  Topic 2: link, time, way, good, al, different, responses, other, questions, client
  Topic 3: important, time, good, human, able, conversation, other, way, sure, feelings

=== K = 5 | c_v = 0.761 ===
  Topic 0: story, time, human, good, other, responses, al, prompt, ior, life
  Topic 1: open, p

In [ ]:
key_u=['u2019s', 'u2019d', 'u2019t', '9s', '9t', 'it']
for x in key_u:
    neg_red=pd.DataFrame(neg_red['Text'].str.replace(x, ''))
    neu_pos_red['Text']=neu_pos_red['Text'].str.replace(x, '')

In [ ]:
textsRedNeg = [row.split() for row in neg_red.Text.tolist()]
dict_for_coh_Red_Neg = Dictionary(textsRedNeg)

textsRedPos = [row.split() for row in neu_pos_red.Text.tolist()]
dict_for_coh_Red_Pos = Dictionary(textsRedPos)

In [ ]:
cohere_numTops(neg_red, dict_for_coh_Red_Neg, textsRedNeg, 2, 7)

NameError: name 'cohere_numTops' is not defined

In [ ]:
cohere_numTops(neu_pos_red, dict_for_coh_Red_Pos, textsRedPos, 2, 7)


=== K = 2 | c_v = 0.768 ===
  Topic 0: open, prompt, questions, link, post, free, image, concerns, action, moderators
  Topic 1: time, questions, human, thoughts, other, way, good, things, self, such

=== K = 3 | c_v = 0.772 ===
  Topic 0: open, prompt, post, link, questions, image, concerns, free, action, moderators
  Topic 1: different, practice, virtual, ideas, time, capable, al, new, techniques, advice
  Topic 2: time, good, questions, way, other, human, thoughts, important, things, responses

=== K = 4 | c_v = 0.764 ===
  Topic 0: open, prompt, questions, post, link, image, concerns, free, action, moderators
  Topic 1: human, role, link, session, time, advice, report, tony, al, client
  Topic 2: good, human, important, time, way, responses, questions, client, other, things
  Topic 3: time, self, thoughts, such, questions, different, way, practice, positive, negative

=== K = 5 | c_v = 0.770 ===
  Topic 0: open, prompt, post, questions, image, concerns, free, action, moderators, c

##Twitter sentiment topics

In [ ]:
cohere_numTops(neg_tweets, dict_for_coh_TWT_Neg, textsTWTNeg, 2, 7)


=== K = 2 | c_v = 0.792 ===
  Topic 0: free, chat, work, bad, basic, close, bitch, masad, amjad, reporter
  Topic 1: open, good, time, real, chat, help, other, use, bad, new

=== K = 3 | c_v = 0.801 ===
  Topic 0: chat, free, way, expensive, bad, art, worse, someone, other, new
  Topic 1: help, chat, good, open, real, mad, issues, much, other, wrong
  Topic 2: open, good, basic, bitch, time, human, close, reporter, masad, amjad

=== K = 4 | c_v = 0.798 ===
  Topic 0: free, chat, expensive, worse, way, new, life, someone, wrong, solopreneur
  Topic 1: real, good, chat, dangerous, issues, mad, new, help, please, thing
  Topic 2: open, basic, bitch, close, reporter, masad, amjad, mnstream, advice, bad
  Topic 3: good, bad, art, work, open, other, stuff, chat, world, model

=== K = 5 | c_v = 0.805 ===
  Topic 0: free, wrong, someone, way, expensive, chat, other, advice, money, everything
  Topic 1: real, good, help, issues, open, something, dangerous, please, everyone, thing
  Topic 2: op

In [ ]:
cohere_numTops(neu_pos_tweets, dict_for_coh_TWT_Pos, textsTWTPos, 2, 7)


=== K = 2 | c_v = 0.728 ===
  Topic 0: new, more, art, prompt, zealy, data, good, news, time, day
  Topic 1: video, chat, best, free, open, 推特账号, 账号, 电报号, 飞机号, enable

=== K = 3 | c_v = 0.726 ===
  Topic 0: data, new, prompt, time, gene, day, good, more, human, year
  Topic 1: chat, 推特账号, best, 电报号, 账号, 飞机号, 谷歌账号, free, tiktok账号, 脸书号
  Topic 2: video, piped, art, zealy, more, care, io, new, si, chat

=== K = 4 | c_v = 0.726 ===
  Topic 0: new, data, time, gene, prompt, good, benefit, year, other, more
  Topic 1: 推特账号, 电报号, 账号, 飞机号, 谷歌账号, tiktok账号, 脸书号, ins号, best, voice
  Topic 2: video, piped, art, zealy, io, si, more, new, care, money
  Topic 3: chat, playback, enable, gender, best, free, app, human, tech, bodycalc

=== K = 5 | c_v = 0.731 ===
  Topic 0: data, new, prompt, time, benefit, good, gene, real, best, therapies
  Topic 1: 推特账号, 电报号, 账号, 飞机号, 谷歌账号, tiktok账号, 脸书号, ins号, best, voice
  Topic 2: video, piped, zealy, io, si, art, more, money, care, join
  Topic 3: chat, gender, 

In [ ]:
nltk.download('words')
tweets_eng_neg=[]
tweet_list=neg_tweets.Text.tolist()
english_vocab = set(w.lower() for w in words.words())
i=0
for text in tweet_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() not in english_vocab) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  tweets_eng_neg.append(text)

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


btc


@

matic
'pessimistic
crypto-
antics
cryptohumor
astro-creep
@
jortega95


onk136
j'
que
j'
goethe
|
🍉
@
inspectornerd

reasons



problems
means

@


retl

@
raleigh
@
makirollofc

babes

generators
girlies
folks
jobs
funko
munter
@
itsmunter
replying

services

deaner444
@
deanehenryson



app
google
tic-

selfcare


freeapp

bgatesisapyscho

🚨💉
%
arrests
wasn
athletes
bs
mrna
experi

’
’
happened…….it
hasn
’
msm
’
questions
greatest

@


structures

shitty
🇵🇸
@

fr
rn…
seán
t.
@
mer__edith
@
sociopathic

researchers

𖤐
@
ghoulglrl

youre


idrc
roy/
@
royscott

replying
witchyscott
ill.
queer🇺🇸
🏳️‍🌈
@
_iggy
@
ineya_veganfood
@
patrickstrud

ones

@



fucked
@
lola19828


quick-
@
mad_in_america



app
interventions
journaling
psychoeducation
positive/negative
madinamerica./2023/12/-…
devuono
@

ctvnewsnorthern
todays
services
mrna
diseases


maracha
@
quitsland

yepindeedagoat
@
alexhaedda
@
im
girls—
ve
relationships
mom
women
mntn
relationships
women


feith
@

infantilizat

In [ ]:
tweets_eng_pos=[]
tweet_list=neu_pos_tweets.Text.tolist()
english_vocab = set(w.lower() for w in words.words())
i=0
for text in tweet_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() not in english_vocab) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  tweets_eng_pos.append(text)

Streaming output truncated to the last 5000 lines.
app
google
tic-



selfcare

freeapp
coolapps

@
luisscarchi


👇
convo
judgement


@


seekers

chats


apps
thought-provoking
@
sciam


artificialintelligence

chats
bit.ly/3p37myg
َlyle
🍭
@


’
pov

nolan
@

safe/expected
answers

panies
tdoc
betterhelp
online

sheinberg
ph.d.
@


chats


apps

programs

programs
unvetted
bit.ly/42ofv1c
randahl
@

endwokeness

nik💥
@
ns123abc




j.earley
one-on-one

neuro
🧠🗺️
@

online
prof.

learnings
n=2400
cbt
pdt
+
thoughts

neuro
|
world-wide.org/seminar/9114
@

nickadobos


ent


@



woods
problems
@

lazlowoodbine42
@
horatioyuletide
@
jtylerhagen


🇵🇸
🍉
@

crulge
@
_eric_reinhart



disorders
plicates
text-based
input/output

utis
@
titletrophy

talking.
instalments


domnname
psycho
domnnameforsale

4open
babyagi
✨
@


brinks
@


snes
’
@
sen7ience

emls

dokja
webshrink
@

services

-powered

varepsilon
@
var_epsilon

tradfischizo
“

”
bro
“
”

ui

@
grc99__

seramoonie
wt


clarke
@






In [ ]:
tweets_engN_df=pd.DataFrame(tweets_eng_neg, columns=['Text'])
tweets_engP_df=pd.DataFrame(tweets_eng_pos, columns=['Text'])

In [ ]:
tweets_engN_df['Text']=tweets_engN_df['Text'].str.lower()
tweets_engN_df=pd.DataFrame(tweets_engN_df.Text.apply(nouns_adj))

tweets_engP_df['Text']=tweets_engP_df['Text'].str.lower()
tweets_engP_df=pd.DataFrame(tweets_engP_df.Text.apply(nouns_adj))

In [ ]:
tweets_engP_df

,Text
0,dev session technology
1,i instant
2,mr tweet test advisement advisement generative...
3,check poe./
4,longevity gene genetic engineering stem rigid ...
...,...
4141,s myriad ethical practical
4142,suicide prevention prospect treatment myriad c...
4143,english myriad ethical practical
4144,paul draw revolution


In [ ]:
neu_pos_tweets

,Unnamed: 0,Text,sentiment
0,0,techforgood 💙 dev khanna ces2024 @ curieuxexpl...,"{'label': 'positive', 'score': 0.8860149383544..."
1,1,~♡ @ replying mikaqvia hugsss 🫂🫂 cuz megumi i ...,"{'label': 'positive', 'score': 0.8713191151618..."
2,2,mr tweet @ renowned test advisement newszo...,"{'label': 'neutral', 'score': 0.8472962379455566}"
3,3,matt ahmann @ check thera- poe./thera-,"{'label': 'neutral', 'score': 0.8349331021308899}"
4,4,𝙲𝚊𝚝𝚊𝚕𝚕𝚊𝚡𝚎𝚛 @ catallaxer “ [ conferences longev...,"{'label': 'neutral', 'score': 0.7989817261695862}"
...,...,...,...
4141,4697,uģijs s šodien myriad ethical practical con...,"{'label': 'neutral', 'score': 0.8168298602104187}"
4142,4698,unisa suicide prevention @ prospect ment...,"{'label': 'neutral', 'score': 0.6521839499473572}"
4143,4699,jazeera english @ ajenglish myriad ethical ...,"{'label': 'neutral', 'score': 0.9171144962310791}"
4144,4700,paul draw @ revolution next aljazeera./ec...,"{'label': 'neutral', 'score': 0.80653315782547}"


In [ ]:
textsTWTNeg = [row.split() for row in tweets_engN_df.Text.tolist()]
dict_for_coh_TWT_Neg = Dictionary(textsTWTNeg)

textsTWTPos = [row.split() for row in tweets_engP_df.Text.tolist()]
dict_for_coh_TWT_Pos = Dictionary(textsTWTPos)

In [ ]:
cohere_numTops(tweets_engN_df, dict_for_coh_TWT_Neg, textsTWTNeg, 2, 7)


=== K = 2 | c_v = 0.778 ===
  Topic 0: bad, wrong, way, other, someone, real, new, many, world, help
  Topic 1: free, good, open, advice, bitch, basic, dangerous, reporter, use, human

=== K = 3 | c_v = 0.796 ===
  Topic 0: real, way, world, bad, time, many, language, whole, worse, wrong
  Topic 1: free, new, advice, good, open, other, human, expensive, guy, playback
  Topic 2: bad, bitch, basic, open, good, reporter, wrong, something, someone, session

=== K = 4 | c_v = 0.787 ===
  Topic 0: way, bad, world, real, many, time, language, gay, good, actual
  Topic 1: free, advice, open, dangerous, new, expensive, good, playback, enable, data
  Topic 2: bitch, basic, reporter, bad, something, wrong, open, someone, everything, good
  Topic 3: good, other, stuff, own, mad, worse, new, human, real, sad

=== K = 5 | c_v = 0.793 ===
  Topic 0: real, bad, way, world, actual, gay, language, good, thing, harvard
  Topic 1: free, advice, dangerous, new, expensive, open, playback, enable, good, way

In [ ]:
cohere_numTops(tweets_engP_df, dict_for_coh_TWT_Pos, textsTWTPos, 2, 7)


=== K = 2 | c_v = 0.754 ===
  Topic 0: free, playback, enable, new, art, video, data, prompt, advice, care
  Topic 1: human, best, good, use, technology, voice, tool, time, life, new

=== K = 3 | c_v = 0.756 ===
  Topic 0: playback, enable, free, new, art, care, advice, year, prompt, girl
  Topic 1: use, voice, technology, human, good, tool, life, real, al, tech
  Topic 2: best, video, free, new, human, gender, good, money, more, time

=== K = 4 | c_v = 0.747 ===
  Topic 0: data, prompt, year, something, open, new, work, world, benefit, chat
  Topic 1: use, voice, good, tool, al, real, technology, conversation, human, session
  Topic 2: best, video, free, gender, money, human, new, good, time, open
  Topic 3: enable, playback, care, new, free, art, platform, news, creator, advice

=== K = 5 | c_v = 0.758 ===
  Topic 0: year, something, open, new, data, world, happy, project, talk, lot
  Topic 1: voice, use, good, al, tool, conversation, real, session, life, human
  Topic 2: best, vide

##pos/neu split

In [ ]:
neu_pos_tweets

,Unnamed: 0,Text,sentiment
0,0,techforgood 💙 dev khanna ces2024 @ curieuxexpl...,"{'label': 'positive', 'score': 0.8860149383544..."
1,1,~♡ @ replying mikaqvia hugsss 🫂🫂 cuz megumi i ...,"{'label': 'positive', 'score': 0.8713191151618..."
2,2,mr tweet @ renowned test advisement newszo...,"{'label': 'neutral', 'score': 0.8472962379455566}"
3,3,matt ahmann @ check thera- poe./thera-,"{'label': 'neutral', 'score': 0.8349331021308899}"
4,4,𝙲𝚊𝚝𝚊𝚕𝚕𝚊𝚡𝚎𝚛 @ catallaxer “ [ conferences longev...,"{'label': 'neutral', 'score': 0.7989817261695862}"
...,...,...,...
4141,4697,uģijs s šodien myriad ethical practical con...,"{'label': 'neutral', 'score': 0.8168298602104187}"
4142,4698,unisa suicide prevention @ prospect ment...,"{'label': 'neutral', 'score': 0.6521839499473572}"
4143,4699,jazeera english @ ajenglish myriad ethical ...,"{'label': 'neutral', 'score': 0.9171144962310791}"
4144,4700,paul draw @ revolution next aljazeera./ec...,"{'label': 'neutral', 'score': 0.80653315782547}"


In [ ]:
import ast


In [ ]:
neu_pos_tweets['sentiment'] = neu_pos_tweets['sentiment'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
neu_pos_red['sentiment'] = neu_pos_red['sentiment'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
neu_pos_yt['sentiment'] = neu_pos_yt['sentiment'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
mask = neu_pos_tweets['sentiment'].str.get('label') == 'neutral'
inverse_mask = ~mask

In [ ]:
neu_tweets = neu_pos_tweets[mask]
pos_tweets = neu_pos_tweets[inverse_mask]

In [ ]:
print(neu_tweets.shape)
print(pos_tweets.shape)

(2565, 3)
(1581, 3)


In [ ]:
mask = neu_pos_red['sentiment'].str.get('label') == 'neutral'
inverse_mask = ~mask

In [ ]:
neu_red = neu_pos_red[mask]
pos_red = neu_pos_red[inverse_mask]

In [ ]:
print(neu_red.shape)
print(pos_red.shape)

(521, 3)
(127, 3)


In [ ]:
mask = neu_pos_yt['sentiment'].str.get('label') == 'neutral'
inverse_mask = ~mask

In [ ]:
neu_yt = neu_pos_yt[mask]
pos_yt = neu_pos_yt[inverse_mask]

In [ ]:
print(neu_yt.shape)
print(pos_yt.shape)

(896, 3)
(769, 3)


###export for later

In [ ]:
neu_yt.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neu.csv")
pos_yt.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_pos.csv")

In [ ]:
neu_red.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_neu.csv")
pos_red.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_pos.csv")

In [ ]:
neu_tweets.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_neu.csv")
pos_tweets.to_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/Twitter/twitter_relevant_pos.csv")

###cohere

In [ ]:
textsYoutubeNeu = [row.split() for row in neu_yt.Text.tolist()]
dict_for_coh_YT_Neu = Dictionary(textsYoutubeNeu)

textsYoutubePos = [row.split() for row in pos_yt.Text.tolist()]
dict_for_coh_YT_Pos = Dictionary(textsYoutubePos)

textsRedNeu = [row.split() for row in neu_red.Text.tolist()]
dict_for_coh_Red_Neu = Dictionary(textsRedNeu)

textsRedPos = [row.split() for row in pos_red.Text.tolist()]
dict_for_coh_Red_Pos = Dictionary(textsRedPos)

textsTWTNeu = [row.split() for row in neu_tweets.Text.tolist()]
dict_for_coh_TWT_Neu = Dictionary(textsTWTNeu)

textsTWTPos = [row.split() for row in pos_tweets.Text.tolist()]
dict_for_coh_TWT_Pos = Dictionary(textsTWTPos)

In [ ]:
len(dict_for_coh_TWT_Pos)

2977

In [ ]:
cohere_numTops(neu_yt, dict_for_coh_YT_Neu, textsYoutubeNeu, 2, 7)


=== K = 2 | c_v = 0.747 ===
  Topic 0: way, good, real, time, other, own, life, many, things, helpful
  Topic 1: way, good, real, something, someone, time, things, lot, more, chat

=== K = 3 | c_v = 0.743 ===
  Topic 0: way, good, time, many, response, something, things, own, more, work
  Topic 1: way, things, good, someone, something, help, time, better, real, lot
  Topic 2: real, good, other, life, time, support, technology, self, own, way

=== K = 4 | c_v = 0.742 ===
  Topic 0: way, good, time, life, prompt, things, many, game, bit, question
  Topic 1: way, good, something, someone, things, help, time, better, more, lot
  Topic 2: real, good, time, self, support, own, other, emotional, way, something
  Topic 3: chat, real, things, better, way, time, other, life, ve, same

=== K = 5 | c_v = 0.763 ===
  Topic 0: way, good, things, trauma, question, real, truth, own, response, self
  Topic 1: way, good, something, time, help, someone, things, better, lot, real
  Topic 2: real, good, o

In [ ]:
cohere_numTops(pos_yt, dict_for_coh_YT_Pos, textsYoutubePos, 2, 7)


=== K = 2 | c_v = 0.774 ===
  Topic 0: good, way, great, things, better, time, more, many, help, ve
  Topic 1: helpful, real, better, life, time, chat, lot, advice, ve, good

=== K = 3 | c_v = 0.771 ===
  Topic 0: good, way, great, time, more, better, things, many, help, own
  Topic 1: life, chat, real, care, ve, advice, lot, much, many, way
  Topic 2: helpful, better, good, great, lot, time, real, ve, something, someone

=== K = 4 | c_v = 0.796 ===
  Topic 0: good, way, great, time, help, things, many, better, bit, lot
  Topic 1: chat, care, advice, life, much, real, things, day, helpful, time
  Topic 2: helpful, better, lot, ve, good, time, something, things, other, years
  Topic 3: good, real, tool, great, better, time, life, many, more, way

=== K = 5 | c_v = 0.769 ===
  Topic 0: way, help, things, problems, time, bad, better, own, great, more
  Topic 1: chat, care, life, much, advice, best, helpful, real, information, way
  Topic 2: helpful, better, ve, lot, great, good, somethin

In [ ]:
cohere_numTops(neu_red, dict_for_coh_Red_Neu, textsRedNeu, 2, 7)


=== K = 2 | c_v = 0.781 ===
  Topic 0: prompt, open, post, questions, concerns, image, free, action, moderators, comment
  Topic 1: time, questions, other, link, responses, thoughts, things, human, way, session

=== K = 3 | c_v = 0.780 ===
  Topic 0: client, good, same, time, treatment, feelings, able, ior, things, anxiety
  Topic 1: questions, link, time, other, responses, thoughts, things, such, advice, way
  Topic 2: open, prompt, post, questions, concerns, free, image, action, moderators, comment

=== K = 4 | c_v = 0.778 ===
  Topic 0: client, good, same, feelings, time, prompt, questions, human, role, example
  Topic 1: link, time, questions, responses, other, al, language, things, such, thoughts
  Topic 2: open, prompt, questions, post, concerns, free, image, action, moderators, discord
  Topic 3: client, other, something, medical, human, anyone, professional, ior, treatment, article

=== K = 5 | c_v = 0.772 ===
  Topic 0: feelings, good, client, same, human, time, example, way,

In [ ]:
cohere_numTops(pos_red, dict_for_coh_Red_Pos, textsRedPos, 2, 7)


=== K = 2 | c_v = 0.745 ===
  Topic 0: good, session, human, way, client, hera, questions, great, thoughts, advice
  Topic 1: good, link, time, life, things, week, diary, lot, story, someone

=== K = 3 | c_v = 0.750 ===
  Topic 0: session, good, thoughts, system, way, data, client, time, prompt, real
  Topic 1: good, link, time, diary, week, things, life, story, ni, someone
  Topic 2: good, hera, human, questions, such, way, professional, great, tool, advice

=== K = 4 | c_v = 0.753 ===
  Topic 0: good, way, able, data, system, thoughts, chat, prompt, real, information
  Topic 1: link, good, time, diary, week, things, ni, lot, mom, crazy
  Topic 2: hera, good, conversation, free, human, client, conversations, questions, fields, sopi
  Topic 3: story, session, human, great, life, good, characters, world, things, self

=== K = 5 | c_v = 0.756 ===
  Topic 0: good, prompt, system, chat, data, information, real, idea, today, way
  Topic 1: link, good, time, diary, week, things, life, ni, l

In [ ]:
tweets_eng_pos=[]
tweet_list=pos_tweets.Text.tolist()
english_vocab = set(w.lower() for w in words.words())
i=0
for text in tweet_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() not in english_vocab) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  tweets_eng_pos.append(text)

techforgood
💙
khanna
ces2024
@
curieuxexplorer

-model
patients
emotions

ones
bit.ly/3s185pi
@
careldr
@
@
@
shi4tech
@
heinzvhoenen
@
@
gvalan
@
drnikolova_rumi
@
sminaev2015
ces
ces24
ination
~♡
@
replying
mikaqvia
hugsss
🫂🫂
cuz
megumi
kagome
hugs
dániel
takács


✨
@

🌟
ine


🎨
-generated
🧑‍🎨
others
👉.me
beancreator
creators
lionell
|
🔎𝕏
|
xar
|
💙
@
mander
vikram
karve
@
selfhelp

worries
👇
wp.me/p1em7-6cr
stories
vikram
karve
pune
blogger
sheen18
@
web3


creators
visitors
revolutionalized
@

toptools
empowers



forbes./sites/lanceeliot/…


/
⚢
@
_ryy2
lets
togetherr🫂🫂🫂
mewen
|
frieren

@

funniest
fatalis

hands
いわたわし
@
tawashi_cb

なんなんこいつｗｗ

bjr
journals
bjr_radiology
bjr

imaging

inations
issam
naqa
bjr
articles
bit.ly/3puywft
highestself.
@
meets

steps



intertwines
tlored

ination
digitalempathy
iraconda
mel_vil


—but
|
artworks
ola


✨
@

🎉

zealy
campgn
contributors

skills
inate
rewards
wins
❤️
detls
👉join
zealy.io/c/
xzealy
rdrop
jyotirgamya
@
jyotirgamya_org


alized

In [ ]:
tweets_eng_neu=[]
tweet_list=neu_tweets.Text.tolist()
i=0
for text in tweet_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() not in english_vocab) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  tweets_eng_neu.append(text)

Streaming output truncated to the last 5000 lines.



updates

@

ethanjkemp
@
sama
github

fast-uptake
topics
—

jokes
@
titletrophy

taking.
bargn
hours


startup

domnname
@



@


fanfictions
mau❄️⛄
@

@


replikq

bros

slabodnick
@


panies


....
dsuke
sasajima
英語学習
@

【
】
知り合いである
’


dsuke
sasajima
英語学習
@

【
ぼんやり
’


bmj

informatics

🔬
human-
ision-making

ision
tumour
bit.ly/44ddcwy
@
lsveikata
inmedicine
digipheno
@





theswaddle./inadequate-me…
domo
@

@
scottjohnson
spreadsheet
cells

deaner444
@
deanehenryson



app
google
tic-



selfcare

freeapp
coolapps
asmr

@



reflections
tweets
thoughts
memes
brnstorm



phd
@

tech-infused



bigdataanalyticsnews./cha…
bigdata
darrel
aasekera
@

tech-infused



catboots_

rowngarnbii
@
shoe0nhead
@
michaelmalice

“

るきの
@
hsgolmskwqfkn6j

´・ω・
安江工務店
株主優待制度を導入
パルｈｄ
ハニーズｈｄ
ファマライズ
サイトリ細研
cytori
技術の特許取得
チェンジｈｄ
rafiq
@

tech-infused



stéphane
bellanger
@

tech-infused



dlvr.it/ss0qj3
fzan
afzal
@






services

resources
exper

In [ ]:
tweets_engNeu_df=pd.DataFrame(tweets_eng_neu, columns=['Text'])
tweets_engPos_df=pd.DataFrame(tweets_eng_pos, columns=['Text'])

In [ ]:
textsTWTNeu = [row.split() for row in tweets_engNeu_df.Text.tolist()]
dict_for_coh_TWT_Neu = Dictionary(textsTWTNeu)

textsTWTPos = [row.split() for row in tweets_engPos_df.Text.tolist()]
dict_for_coh_TWT_Pos = Dictionary(textsTWTPos)

In [ ]:
cohere_numTops(tweets_engPos_df, dict_for_coh_TWT_Pos, textsTWTPos, 2, 7)


=== K = 2 | c_v = 0.785 ===
  Topic 0: video, platform, free, creator, best, money, time, visitor, create, great
  Topic 1: art, best, chat, new, good, gender, more, care, news, io

=== K = 3 | c_v = 0.768 ===
  Topic 0: video, money, create, free, data, content, other, time, tool, ways
  Topic 1: art, good, gender, more, io, new, best, chat, big, prize
  Topic 2: best, platform, new, chat, creator, care, visitor, today, al, technology

=== K = 4 | c_v = 0.780 ===
  Topic 0: time, today, art, tool, more, best, work, life, enable, human
  Topic 1: art, good, new, sexy, io, alert, best, join, more, human
  Topic 2: platform, best, care, new, creator, chat, visitor, future, technology, more
  Topic 3: video, gender, best, chat, money, data, free, create, great, confidential

=== K = 5 | c_v = 0.780 ===
  Topic 0: time, year, happy, tool, work, best, help, life, use, way
  Topic 1: good, new, io, art, join, best, prize, tool, conversation, more
  Topic 2: platform, visitor, creator, best,

In [ ]:
cohere_numTops(tweets_engNeu_df, dict_for_coh_TWT_Neu, textsTWTNeu, 2, 7)


=== K = 2 | c_v = 0.745 ===
  Topic 0: al, new, tool, human, data, use, care, prompt, voice, open
  Topic 1: free, chat, playback, enable, advice, good, best, real, open, technology

=== K = 3 | c_v = 0.749 ===
  Topic 0: care, prompt, data, new, benefit, use, nick, ethical, research, anyone
  Topic 1: free, chat, playback, enable, advice, real, best, dialogue, relationship, technology
  Topic 2: open, human, voice, al, tool, conversation, good, stress, next, use

=== K = 4 | c_v = 0.760 ===
  Topic 0: prompt, care, data, benefit, new, nick, ethical, content, beginner, analytics
  Topic 1: free, playback, enable, advice, chat, real, dialogue, relationship, play, cools
  Topic 2: human, voice, tool, al, good, open, conversation, next, stress, productivity
  Topic 3: chat, open, free, best, session, use, world, time, technology, professional

=== K = 5 | c_v = 0.761 ===
  Topic 0: prompt, benefit, nick, new, ethical, beginner, content, care, analytics, data
  Topic 1: free, playback, en

#Looking at Youtube Video Titles

In [ ]:
yt_titles=pd.read_csv("/content/drive/MyDrive/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_comment_titles.csv")


In [ ]:
yt_titles.rename(columns={'Video_title': 'Text'}, inplace=True)


In [ ]:
yt_titles

,Unnamed: 0,Video_Id,Text
0,0,TeJppS_wXOk,Things to consider if you’re using ChatGPT as ...
1,1,TeJppS_wXOk,Things to consider if you’re using ChatGPT as ...
2,2,TeJppS_wXOk,Things to consider if you’re using ChatGPT as ...
3,3,TeJppS_wXOk,Things to consider if you’re using ChatGPT as ...
4,4,TeJppS_wXOk,Things to consider if you’re using ChatGPT as ...
...,...,...,...
4429,4429,3BgdhHXJ-CU,Can AI Replace Therapists? | Psychiatrist Expl...
4430,4430,3BgdhHXJ-CU,Can AI Replace Therapists? | Psychiatrist Expl...
4431,4431,3BgdhHXJ-CU,Can AI Replace Therapists? | Psychiatrist Expl...
4432,4432,3BgdhHXJ-CU,Can AI Replace Therapists? | Psychiatrist Expl...


In [ ]:
unique_df = pd.DataFrame(yt_titles['Text'].drop_duplicates())


In [ ]:
unique_df.size

49

In [ ]:
unique_df

,Text
0,Things to consider if you’re using ChatGPT as ...
12,ChatGPT is Everyone&#39;s Therapist Now?
14,I&#39;ve Been Using ChatGPT as a Therapist
72,My take on AI Therapy
90,Is ChatGPT a Better Therapist Than Me?!
111,ChatGPT Therapy is Dangerous
208,Is AI therapy a horrible idea?
239,Why AI Will Replace Your Therapist
298,Can AI Replace Therapists? | Psychiatrist Expl...
492,Therapist vs. Artificial Intelligence - I answ...


In [ ]:
unique_df.Text.apply(len).mean()

np.float64(50.08163265306123)

In [ ]:
unique_df=pd.DataFrame(unique_df.Text.str.lower())

In [ ]:
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
keys_punc=['?', '!', '#', '&', '.', ';',':', '(',')', ',']
for x in keys_punc:
    unique_df['Text']=unique_df['Text'].str.replace(x, ' ')

In [ ]:
title_stop=[]
title_list=unique_df.Text.tolist()
english_vocab = set(w.lower() for w in words.words())
i=0
for text in title_list:
  for word in text.split(' '):
    #if i<10:
      #print(word)
    if (word.lower() in stop_words) or not(word.isalpha()):
      print(word)
      text=text.replace(word, '')
  title_stop.append(text)

to
if
you’re
as
your
👀
is

39
s
now

i

39
ve
been
as
a
my
on
is
a
than
me


is
is
a

why
will
your
can

|

-
i
your


can
with

should
you
for

|
isn’t
your


why
it
as
is
why
is
an
chat-gpt
as
a
can
be
your

|
the

is
not
your








as
a
is
no
than
an
how
to
be
your
do
you
as
a

when
is
your
therapist…
😅😭😂






how
to
a
is
to
as
a

will

as
a





can
your


to
the




can
a
be
your

w/

|
being
isn

39
t
the
y

39
all
it
to
be




more

as
your

has
a

how
to
into
your
—


is
my

you
with
your

3
a
as
your

stop‼️
should
you
for

here

39
s
what
a
is
my



i
into
my

here

39
s
how

why
are
more
to
for

is
a

🤔
i
be
my
therapist…
and
it
my
should
you
as
a






about
@thediaryofaceo
the
for
|
|
the
of


In [ ]:
unique_df

,Text
0,things to consider if you’re using chatgpt as ...
12,chatgpt is everyone 39 s therapist now
14,i 39 ve been using chatgpt as a therapist
72,my take on ai therapy
90,is chatgpt a better therapist than me
111,chatgpt therapy is dangerous
208,is ai therapy a horrible idea
239,why ai will replace your therapist
298,can ai replace therapists | psychiatrist expl...
492,therapist vs artificial intelligence - i answ...


In [ ]:
title_stop_df=pd.DataFrame(title_stop, columns=['Text'])

In [ ]:
unique_na_df=pd.DataFrame(unique_df.Text.apply(nouns_adj))

In [ ]:
unique_na_df

,Text
0,things chatgpt therapist 👀
12,chatgpt everyone s therapist
14,i ve chatgpt therapist
72,take ai therapy
90,better therapist
111,chatgpt therapy dangerous
208,ai therapy horrible idea
239,therapist
298,therapists | psychiatrist explains
492,therapist vs artificial intelligence i questio...


In [ ]:
keys_titles=['ai', 'chatgpt', 'therapist', 'therapy']
for x in keys_titles:
    unique_na_df['Text']=unique_na_df['Text'].str.replace(x, '')
    title_stop_df['Text']=title_stop_df['Text'].str.replace(x, '')


In [ ]:
print(unique_na_df.Text.apply(len).mean())
print(title_stop_df.Text.apply(len).mean())

17.081632653061224
32.326530612244895


In [ ]:
text_title_na = [row.split() for row in unique_na_df.Text.tolist()]
dict_for_coh_title_na = Dictionary(text_title_na)

text_title_stop = [row.split() for row in title_stop_df.Text.tolist()]
dict_for_coh_title_stop = Dictionary(text_title_stop)

In [ ]:
cohere_numTops(unique_na_df, dict_for_coh_title_na, text_title_na, 2, 7)


=== K = 2 | c_v = 0.740 ===
  Topic 0: techtok, quot, explns, take, personal, dangerous, vs, tech, carterpcs, techfacts
  Topic 1: dr, amp, gpt, ft, health, micaela, honda, test, mental, vs

=== K = 3 | c_v = 0.755 ===
  Topic 0: techtok, quot, take, dangerous, health, dive, jacobson, nick, deep, carterpcs
  Topic 1: gpt, amp, dangerous, comedy, apps, funny, shorts, reviews, loneliness, gen
  Topic 2: dr, ft, vs, amp, micaela, honda, test, personal, explns, health

=== K = 4 | c_v = 0.764 ===
  Topic 0: dangerous, techtok, chatbot, dive, jacobson, nick, deep, techfacts, carterpcs, tech
  Topic 1: gpt, health, apps, shorts, comedy, funny, psychologist, loneliness, reviews, romance
  Topic 2: dr, ft, vs, amp, micaela, honda, test, explns, mental, real
  Topic 3: quot, personal, take, gpt, dangerous, isn, warning, sam, thinks, altman

=== K = 5 | c_v = 0.750 ===
  Topic 0: dangerous, take, chatbot, dive, nick, jacobson, deep, coach, thinks, ultimate
  Topic 1: gpt, shorts, comedy, romanc

In [ ]:
cohere_numTops(title_stop_df, dict_for_coh_title_stop, text_title_stop, 2, 7)


=== K = 2 | c_v = 0.819 ===
  Topic 0: therapt, therpt, chtgpt, using, therapst, thn, quot, take, dangerous, mental
  Topic 1: chtgpt, therpist, use, dr, vs, replace, amp, ft, gpt, chat

=== K = 3 | c_v = 0.781 ===
  Topic 0: therapt, therapst, quot, dangerous, take, explns, vs, using, answer, mentalhealth
  Topic 1: therpist, chtgpt, use, dr, vs, using, amp, ft, chtbot, therpy
  Topic 2: chtgpt, therpt, replace, health, mental, techtok, gpt, chat, dr, thn

=== K = 4 | c_v = 0.772 ===
  Topic 0: therapt, quot, dangerous, take, explns, vs, using, therapst, chat, gpt
  Topic 1: therpist, use, chtgpt, vs, using, amp, ft, dr, therpy, honda
  Topic 2: replace, therpist, dr, chtgpt, health, mental, techtok, gpt, chat, therapt
  Topic 3: chtgpt, therpt, thn, using, seeing, worse, ctul, better, relatonshp, let

=== K = 5 | c_v = 0.781 ===
  Topic 0: therapt, dangerous, take, using, therapst, chat, gpt, vs, answer, mentalhealth
  Topic 1: vs, amp, dr, ft, therpy, honda, micaela, chtbot, gen, c